# InkMind — GPT-2 Text Generation

**Owner:** Member 4 (GPT-2 Generation)

This notebook shows how GPT-2 turns a prompt into new text and how the
decoding strategy changes the output:

1. Prompt → tokens → token IDs (reusing the GPT-2 tokenizer from the exploration notebook)
2. Autoregressive generation, one token at a time
3. Decoding strategies: greedy, random sampling, temperature, top-k, top-p
4. Experiments on the **same prompts** with different settings
5. Reproducibility with random seeds

The model is the same pretrained `gpt2` (124M) used in
`GPT2_Exploration_(1).ipynb`, loaded with the same Hugging Face classes.
The reusable code lives in the `inkmind/` package so the comparison notebook
(Task 5) and the Streamlit app use exactly the same generation loop.

## 1. Setup

In [ ]:
# --- Setup: works in Google Colab and locally (VS Code / Jupyter) ---------
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nalinegwizaigihozo/Ink-mind_group2.git"

if IN_COLAB:
    !pip install -q transformers
    REPO_DIR = "/content/Ink-mind_group2"
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    # If cloning fails (e.g. private repo), upload the `inkmind/` folder and
    # `data/PoetryFoundationData.csv` into /content/Ink-mind_group2 manually.
else:
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

RESULTS_DIR = os.path.join(REPO_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Running in Colab:", IN_COLAB)
print("Repository dir  :", REPO_DIR)

In [ ]:
import time
import textwrap

import torch
import pandas as pd
import matplotlib.pyplot as plt

from inkmind import GPT2Adapter, generate, load_gpt2, next_token_distribution
from inkmind.comparison import add_text_metrics, environment_info
from inkmind.plotting import MODEL_COLORS, plot_metric_vs_param, style_axes

pd.set_option("display.max_colwidth", 120)
environment_info()

## 2. Load GPT-2

Same model and tokenizer as the exploration notebook (`GPT2Tokenizer`,
`GPT2LMHeadModel`, pretrained `"gpt2"`). The model is put in evaluation mode
and moved to the GPU when one is available.

The load time includes downloading the weights the first time.

In [ ]:
model, tokenizer, gpt2_info = load_gpt2("gpt2")
device = next(model.parameters()).device

gpt2 = GPT2Adapter(model, tokenizer)

print("Device        :", device)
print("Load time (s) :", round(gpt2_info["load_time_s"], 2))
print("Context length:", gpt2.context_length, "tokens")
print("EOS token     :", repr(tokenizer.eos_token), "id =", tokenizer.eos_token_id)

## 3. How GPT-2 generates text

GPT-2 is **autoregressive**: it only ever predicts *one* next token.
To produce a longer text it repeats the same step:

```
prompt ──tokenize──► [token IDs] ──► GPT-2 ──► logits for the next token
                          ▲                         │
                          │            choose one token (greedy / sampling)
                          └──────── append ◄────────┘
                    ... repeat until max length (or <|endoftext|>) ... ──decode──► text
```

The cell below runs the loop **by hand** for 5 steps, printing the 5 most
likely candidates at every step and picking the most likely one (greedy).

In [ ]:
prompt = "Artificial intelligence is"

input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
print("Prompt       :", prompt)
print("Prompt tokens:", tokenizer.convert_ids_to_tokens(input_ids[0]))
print("Prompt IDs   :", input_ids[0].tolist())
print()

ids = input_ids
for step in range(5):
    with torch.no_grad():
        logits = model(ids).logits[:, -1, :]          # scores for the NEXT token only

    probs = torch.softmax(logits, dim=-1)[0]
    top_probs, top_ids = torch.topk(probs, 5)
    candidates = ", ".join(
        f"{tokenizer.decode([i])!r} {p:.2f}" for p, i in zip(top_probs.tolist(), top_ids.tolist())
    )

    next_id = top_ids[:1].view(1, 1)                   # greedy: take the most likely token
    ids = torch.cat([ids, next_id], dim=1)            # append and repeat
    print(f"step {step + 1}: [{candidates}] -> chose {tokenizer.decode(next_id[0])!r}")

print("\nGenerated IDs:", ids[0, input_ids.shape[1]:].tolist())
print("Decoded text :", tokenizer.decode(ids[0]))

### 3.1 Check: our loop vs Hugging Face `model.generate`

`inkmind.generate()` wraps the loop above (plus sampling, timing and
stopping rules). For greedy decoding it must produce exactly the same text
as Hugging Face's built-in `model.generate(do_sample=False)`.

In [ ]:
ours = generate(gpt2, prompt, max_new_tokens=30, strategy="greedy")

enc = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    hf_ids = model.generate(**enc, max_new_tokens=30, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
hf_text = tokenizer.decode(hf_ids[0, enc["input_ids"].shape[1]:])

print("inkmind.generate :", repr(ours.completion))
print("model.generate   :", repr(hf_text))
print("Identical        :", ours.completion == hf_text)

## 4. Decoding strategies

At every step GPT-2 outputs a score (logit) for each of its 50,257 tokens.
How we turn these scores into one chosen token is the *decoding strategy*:

| Strategy | Rule | Effect |
|---|---|---|
| **Greedy** | always take the most likely token | deterministic; often repeats itself |
| **Random sampling** | sample from the full softmax distribution | varied, but can pick very unlikely tokens |
| **Temperature T** | divide logits by T before softmax | T < 1 sharpens (safer), T > 1 flattens (more random) |
| **Top-k** | keep only the k most likely tokens, renormalise, sample | cuts off the long tail of unlikely tokens |
| **Top-p (nucleus)** | keep the smallest set of tokens whose probability adds up to p | the number of candidates adapts to how confident the model is |

In `inkmind.generation.filter_logits` the order is: temperature → top-k → top-p → softmax → sample
(the same order Hugging Face uses).

The chart shows how temperature reshapes the next-token distribution for one prompt.

In [ ]:
demo_prompt = "Once upon a time"
demo_ids = torch.tensor([tokenizer.encode(demo_prompt)], device=device)
demo_logits = gpt2.next_token_logits(demo_ids)

top_ids = torch.topk(demo_logits[0], 12).indices
labels = [repr(tokenizer.decode([i])) for i in top_ids.tolist()]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, T in zip(axes, [0.5, 1.0, 1.5]):
    p = next_token_distribution(demo_logits, temperature=T)[0, top_ids].cpu()
    ax.bar(range(len(labels)), p, color=MODEL_COLORS["GPT-2"], width=0.6)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_title(f"temperature = {T}")
    style_axes(ax)
axes[0].set_ylabel("probability")
fig.suptitle(f"GPT-2 next-token probabilities after {demo_prompt!r} (12 most likely tokens)")
fig.tight_layout()
plt.show()

In [ ]:
# How many candidate tokens survive each filter (same prompt, same step)
settings = {
    "no filter (T=1.0)": {},
    "T=0.5":             dict(temperature=0.5),
    "top-k=5":           dict(top_k=5),
    "top-k=40":          dict(top_k=40),
    "top-p=0.5":         dict(top_p=0.5),
    "top-p=0.9":         dict(top_p=0.9),
    "top-p=0.9, T=0.7":  dict(top_p=0.9, temperature=0.7),
}

rows = []
for name, kw in settings.items():
    p = next_token_distribution(demo_logits, **kw)[0]
    rows.append({
        "setting": name,
        "candidate tokens (prob > 0)": int((p > 0).sum()),
        "prob of most likely token": round(p.max().item(), 3),
    })
pd.DataFrame(rows).set_index("setting")

## 5. Generation experiments

All experiments use the **same four prompts** and the same seeds, so the only
thing that changes between rows is the generation setting.

Each run records: prompt, method, temperature, top-k, top-p, seed, number of
prompt / generated tokens, generated text, and timing (tokenization,
generation loop, decoding).

**Timing note:** the generation loop recomputes the full forward pass at every
step (no key/value cache) so that it is identical to the TinyGPT loop used in
the comparison notebook. Absolute speeds are therefore lower than Hugging
Face's cached `model.generate`, and they depend on the hardware (Colab GPU vs CPU).

In [ ]:
PROMPTS = [
    "Artificial intelligence is",
    "The future of technology is",
    "Machine learning can",
    "Once upon a time",
]
SEEDS = [0, 1, 2]
MAX_NEW_TOKENS = 40

def run_experiment(name, configs, seeds=SEEDS, max_new_tokens=MAX_NEW_TOKENS, prompts=PROMPTS):
    """Run every config on every prompt (and seed); return one row per output."""
    rows = []
    for prompt in prompts:
        for label, cfg in configs.items():
            # greedy is deterministic -> one run is enough
            run_seeds = seeds[:1] if cfg.get("strategy") == "greedy" else seeds
            for seed in run_seeds:
                r = generate(gpt2, prompt, max_new_tokens=max_new_tokens, seed=seed, **cfg)
                rows.append({"experiment": name, "setting": label, **r.to_dict()})
    return add_text_metrics(pd.DataFrame(rows))

def show_outputs(df, seed=0, width=100):
    """Print the generated text for each prompt, setting by setting."""
    for prompt, g in df[(df["seed"] == seed)].groupby("prompt", sort=False):
        print("=" * width)
        print(f"PROMPT: {prompt!r}")
        for _, row in g.iterrows():
            text = textwrap.fill(row["completion"].replace("\n", " ⏎ "), width - 4,
                                 initial_indent="    ", subsequent_indent="    ")
            print(f"\n  [{row['setting']}]  ({row['generated_tokens']} tokens, {row['generation_time_s']:.2f}s)")
            print(text)

### 5.1 Experiment A — decoding methods

Greedy vs pure random sampling vs top-k vs top-p, with a fixed seed.

In [ ]:
methods = {
    "greedy":                dict(strategy="greedy"),
    "random sampling T=1.0": dict(strategy="sample", temperature=1.0),
    "top-k=40, T=1.0":       dict(strategy="sample", temperature=1.0, top_k=40),
    "top-p=0.9, T=1.0":      dict(strategy="sample", temperature=1.0, top_p=0.9),
}
exp_methods = run_experiment("methods", methods)
show_outputs(exp_methods, seed=0)

In [ ]:
cols = ["distinct_1", "distinct_2", "repeated_3gram_rate", "generated_tokens", "tokens_per_s"]
exp_methods.groupby("setting", sort=False)[cols].mean().round(3)

**Reading the metrics** (computed on the decoded words of each output):

- `distinct_1` / `distinct_2`: unique words / word-pairs divided by total — higher means more varied wording.
- `repeated_3gram_rate`: share of 3-word sequences that already appeared earlier in the same output — higher means more looping.

These are simple automatic indicators of repetition and variety, **not** measures of quality or coherence.

**Why do some sampled outputs start with the same words?** With the same seed,
every setting draws the *same random numbers*. When two settings still give the
chosen token enough probability, they pick the same token, so outputs share a
beginning and only split where the filtering actually changes the choice. This
is intentional: it makes the comparison controlled, so differences come from
the setting and not from luck.

### 5.2 Experiment B — temperature

Pure sampling (no top-k / top-p) with temperature 0.3 → 1.5.

In [ ]:
temperatures = [0.3, 0.7, 1.0, 1.5]
exp_temp = run_experiment("temperature", {
    f"T={t}": dict(strategy="sample", temperature=t) for t in temperatures
})
exp_temp["temperature_value"] = exp_temp["temperature"]
show_outputs(exp_temp, seed=0)

In [ ]:
fig = plot_metric_vs_param(
    exp_temp, "temperature_value",
    ["distinct_2", "repeated_3gram_rate"],
    ["Distinct-2 (higher = more varied)", "Repeated 3-gram rate (higher = more looping)"],
    xlabel="temperature",
    suptitle=f"GPT-2: effect of temperature (mean over {len(PROMPTS)} prompts x {len(SEEDS)} seeds)",
)
plt.show()
exp_temp.groupby("setting", sort=False)[cols].mean().round(3)

### 5.3 Experiment C — top-k

Temperature fixed at 1.0; k from 1 (equivalent to greedy) to 200, plus no limit.

In [ ]:
top_ks = [1, 5, 40, 200, None]
exp_topk = run_experiment("top-k", {
    f"top-k={k if k else 'all'}": dict(strategy="sample", temperature=1.0, top_k=k) for k in top_ks
})
show_outputs(exp_topk, seed=0)

In [ ]:
exp_topk.groupby("setting", sort=False)[cols].mean().round(3)

### 5.4 Experiment D — top-p (nucleus)

Temperature fixed at 1.0; p from 0.1 (very conservative) to 1.0 (no filtering).

In [ ]:
top_ps = [0.1, 0.5, 0.9, 1.0]
exp_topp = run_experiment("top-p", {
    f"top-p={p}": dict(strategy="sample", temperature=1.0, top_p=p) for p in top_ps
})
exp_topp["top_p_value"] = exp_topp["top_p"]
show_outputs(exp_topp, seed=0)

In [ ]:
fig = plot_metric_vs_param(
    exp_topp, "top_p_value",
    ["distinct_2", "repeated_3gram_rate"],
    ["Distinct-2 (higher = more varied)", "Repeated 3-gram rate (higher = more looping)"],
    xlabel="top-p",
    suptitle=f"GPT-2: effect of top-p (T=1.0, mean over {len(PROMPTS)} prompts x {len(SEEDS)} seeds)",
)
plt.show()
exp_topp.groupby("setting", sort=False)[cols].mean().round(3)

### 5.5 Experiment E — maximum generated tokens

Same setting (top-p=0.9, T=0.8, seed 0), different `max_new_tokens`.
Because every step re-reads the whole sequence (no cache), each new token
costs a little more than the previous one.

In [ ]:
lengths = [10, 25, 50, 100]
rows = []
for n in lengths:
    df_n = run_experiment("max length",
                          {f"max_new_tokens={n}": dict(strategy="sample", temperature=0.8, top_p=0.9)},
                          seeds=[0], max_new_tokens=n)
    rows.append(df_n)
exp_len = pd.concat(rows, ignore_index=True)

for prompt in PROMPTS[:1]:
    for _, row in exp_len[exp_len["prompt"] == prompt].iterrows():
        print(f"[{row['setting']}] {row['full_text']!r}\n")

len_summary = exp_len.groupby("max_new_tokens")[["generated_tokens", "generation_time_s", "tokens_per_s"]].mean()

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(len_summary.index, len_summary["generation_time_s"], color=MODEL_COLORS["GPT-2"],
        linewidth=2, marker="o", markersize=7, markeredgecolor="white", markeredgewidth=1.5)
ax.set_xlabel("max_new_tokens")
ax.set_ylabel("generation time (s)")
ax.set_title(f"GPT-2 generation time vs length ({device.type}, mean of {len(PROMPTS)} prompts)")
style_axes(ax)
fig.tight_layout()
plt.show()
len_summary.round(3)

### 5.6 Reproducibility with seeds

Sampling is random, so `generate(..., seed=s)` seeds a dedicated
`torch.Generator`. The same seed gives the same text; a different seed gives a
different text. (Results are reproducible on the same hardware and library
versions; GPU vs CPU arithmetic can differ slightly.)

In [ ]:
cfg = dict(strategy="sample", temperature=0.8, top_p=0.9, max_new_tokens=30)
a = generate(gpt2, PROMPTS[3], seed=123, **cfg)
b = generate(gpt2, PROMPTS[3], seed=123, **cfg)
c = generate(gpt2, PROMPTS[3], seed=124, **cfg)

print("seed 123 (run 1):", repr(a.completion))
print("seed 123 (run 2):", repr(b.completion))
print("seed 124        :", repr(c.completion))
print()
print("Same seed -> identical text :", a.completion == b.completion)
print("Different seed -> different :", a.completion != c.completion)

## 6. Save all results

In [ ]:
DOWNLOAD_RESULTS = False   # set True in Colab to download the CSV

all_results = pd.concat([exp_methods, exp_temp, exp_topk, exp_topp, exp_len], ignore_index=True)
all_results.insert(0, "run_device", device.type)

keep = ["run_device", "experiment", "setting", "prompt", "strategy", "temperature", "top_k", "top_p",
        "seed", "max_new_tokens", "prompt_tokens", "generated_tokens", "completion", "stop_reason",
        "tokenize_time_s", "generation_time_s", "decode_time_s", "total_time_s", "tokens_per_s",
        "distinct_1", "distinct_2", "repeated_3gram_rate"]
results_path = os.path.join(RESULTS_DIR, "task4_gpt2_generation_results.csv")
all_results[keep].to_csv(results_path, index=False)
print("Saved", len(all_results), "generations to", results_path)

if IN_COLAB and DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(results_path)

## 7. What to look for in the results

Use the outputs and tables above (not this text) as the evidence. Typical
behaviour reported for GPT-2, which these experiments let you check:

- **Greedy** is deterministic and tends to fall into loops (a high
  repeated 3-gram rate) once it has produced a phrase it finds likely.
- **Low temperature / small top-k / small top-p** behave close to greedy: safer
  and more repetitive.
- **High temperature (e.g. 1.5) or unfiltered sampling** gives the most varied
  wording (high distinct-2) but is more likely to drift off-topic or produce odd words.
- **Top-p** adapts the number of candidates to the model's confidence (see the
  candidate-count table in section 4), which is why moderate settings such as
  p = 0.9 with T ≈ 0.8 are common defaults.
- **Length:** generation time grows with `max_new_tokens`, and slightly faster
  than linearly here because the loop has no cache.

**Limitations:** 4 prompts × 3 seeds is a small sample, the automatic metrics
only measure repetition and variety, and the outputs reflect the pretrained
GPT-2 (web text), not a poetry fine-tuned model.